# Kurtosis

One last pair of indistinguishable-looking games: a plain ±\$1 coin flip, and a game that is *almost always* ±10 cents but once in a while ±\$10. Same mean, same variance, same (zero) skewness — the first three moments all fail. The **fourth moment** finally separates them, and its standardized version is the **kurtosis**: $E\big[\big(\frac{X-\mu}{\sigma}\big)^4\big]$, the measure of **tail thickness**. With this, the toolkit of this lesson is complete: mean, variance, skewness, kurtosis. It is based on the eleventh video of Lesson 3 (Describing Distributions).

## 1. Introduction

Two games:

- **Game 1:** win \$1 with probability $\tfrac12$, lose \$1 with probability $\tfrac12$ — the fair coin toss we have seen many times.
- **Game 2:** win or lose **10 cents**, each with probability $\tfrac{100}{202}$ — a much more conservative game — *except* there is a tiny $\tfrac{1}{202}$ chance of winning **\$10**, and the same of losing **\$10**.

Which game is riskier? Game 2 feels safer (dimes instead of dollars) yet scarier (the rare \$10 swing). As we will compute, the mean, the variance, *and* the skewness are all identical — so if "risk = variance," the games are equally risky. But they are clearly different, and the difference lives **in the tails**. Capturing it needs the fourth moment.

This matters beyond games: financial returns, insurance claims, and network delays often look "normal-ish" in the middle while hiding rare extreme events — exactly what variance underweights and kurtosis detects.

After completing this notebook you will be able to:

- Construct distributions that agree on mean, variance, and skewness yet differ drastically.
- Compute the fourth moment and the **kurtosis** $E\big[\big(\frac{X-\mu}{\sigma}\big)^4\big]$.
- Interpret kurtosis as tail thickness: thin tails → small kurtosis, thick tails → large kurtosis.
- Summarize a distribution with the full four-descriptor toolkit.

## 2. Theory

### The first three moments all fail

**Expected value.** Both games are symmetric around 0, so $E[X_1] = E[X_2] = 0$. No help.

**Variance.** Game 1: $\mathrm{Var}(X_1) = \tfrac12(-1)^2 + \tfrac12(1)^2 = 1$. Game 2 (already centered, so variance is just $E[X_2^2]$):

$$
E[X_2^2] = \tfrac{100}{202}(-0.1)^2 + \tfrac{100}{202}(0.1)^2 + \tfrac{1}{202}(-10)^2 + \tfrac{1}{202}(10)^2
= \tfrac{1}{202} + \tfrac{1}{202} + \tfrac{100}{202} + \tfrac{100}{202} = \tfrac{202}{202} = 1
$$

Believe it or not: **the exact same variance.** By this measure neither game is riskier, and the standard deviations match too. (The numbers $\tfrac{100}{202}$ and $\tfrac{1}{202}$ were chosen precisely to make this happen.)

**Skewness.** Both distributions are symmetric around their center, and symmetric distributions have skewness 0. Also no help.

### The fourth moment

First moment: 0 and 0. Second: 1 and 1. Third: 0 and 0. So — what about the **fourth**?

$$
E[X_1^4] = \tfrac12(-1)^4 + \tfrac12(1)^4 = 1
$$

$$
E[X_2^4] = \tfrac{100}{202}(0.1)^4 \cdot 2 + \tfrac{1}{202}(10)^4 \cdot 2
= \tfrac{0.02}{202} + \tfrac{20{,}000}{202} \approx 99.01
$$

**Huge.** Why? Because $(\pm 10)^4 = 10{,}000$: raising to the fourth power amplifies far-away values so strongly that even a $\tfrac{1}{202}$ probability cannot hide them. When a distribution has values very far from the center — even with tiny probabilities — $E[X^4]$ captures it.

### Kurtosis: the standardized fourth moment

As with skewness, the raw moment still depends on units, so standardize first. The **kurtosis** is

$$
\text{kurtosis} = E\left[\left(\frac{X - \mu}{\sigma}\right)^4\right]
$$

Interpretation:

- **Thin tails** (all values close to the center, extremes essentially absent) → **small kurtosis**.
- **Thick tails** (rare but very distant values) → **large kurtosis**.

Two distributions can share a variance while one hides big rare deviations — the variance can't pick that up well, because a thick-tailed distribution can compensate with a skinnier middle. Kurtosis is the far more sensitive instrument for tail thickness, because the fourth power weights the extremes so heavily.

(Useful benchmark from the moments notebook: the normal distribution has kurtosis exactly **3**. Distributions are often compared against it — "excess kurtosis" is kurtosis minus 3.)

### The complete toolkit

You now have four ways to tell distributions apart, one per moment:

| descriptor | built from | measures |
|---|---|---|
| **expected value** | 1st moment | center |
| **variance / std** | 2nd (central) moment | spread |
| **skewness** | 3rd (standardized) moment | asymmetry |
| **kurtosis** | 4th (standardized) moment | tail thickness |

With these four tools you can say a lot about any probability distribution.

## 3. Worked Examples

### Example 1: the kurtosis of the two games

Both games have $\mu = 0$ and $\sigma = 1$, so the standardization changes nothing and the kurtosis *is* the fourth moment:

$$
\text{kurt}(X_1) = 1 \qquad \text{kurt}(X_2) = 99.01
$$

Game 1 has the smallest possible kurtosis (a two-point symmetric distribution has *no* tails at all); game 2's rare ±\$10 outcomes give it enormously thick tails for its variance.

### Example 2: benchmark shapes

- **Coin (±1):** kurtosis 1 — the minimum possible.
- **Uniform:** kurtosis 1.8 — no tails, flat body.
- **Normal:** kurtosis 3 — the standard reference.
- **Laplace (double exponential):** kurtosis 6 — noticeably thick tails.

Same variance can be arranged for all four; the kurtosis ladder orders them by how much of that variance comes from rare large deviations.

### Example 3: reading a risk report

Two portfolios both report "daily return: mean 0%, std 1%". Portfolio A has kurtosis ≈ 3 (normal-like), portfolio B has kurtosis ≈ 30. B's typical days are *calmer* than A's — but its variance budget is spent on rare violent moves. Same variance, very different risk. This is why heavy-tail awareness matters in finance.

## 4. Python Examples

First the imports (with the numpy 2.x compatibility shim used throughout this project).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# numpy < 2 fallback: np.trapezoid was called np.trapz
if not hasattr(np, "trapezoid"):
    np.trapezoid = np.trapz

rng = np.random.default_rng(42)

### The two games: all four descriptors

In [ ]:
def describe(values, pmf, name):
    mu = np.sum(pmf * values)
    var = np.sum(pmf * (values - mu)**2)
    sigma = np.sqrt(var)
    skew = np.sum(pmf * ((values - mu) / sigma)**3)
    kurt = np.sum(pmf * ((values - mu) / sigma)**4)
    print(f"{name:<8} mean = {mu:>4.1f}   var = {var:>6.4f}   "
          f"skew = {skew:>4.1f}   kurtosis = {kurt:>6.2f}")

game1_values = np.array([-1.0, 1.0])
game1_pmf = np.array([0.5, 0.5])

game2_values = np.array([-10.0, -0.1, 0.1, 10.0])
game2_pmf = np.array([1, 100, 100, 1]) / 202

describe(game1_values, game1_pmf, "game 1")
describe(game2_values, game2_pmf, "game 2")

print("\nMean, variance, and skewness are identical -- only the kurtosis")
print("(4th standardized moment: 1 vs 99.01) separates the games.")

### Watching the games differ in real life

Simulate both games: same long-run average, same variance — but game 2's trajectory is eerily flat until a \$10 shock hits.

In [ ]:
rng = np.random.default_rng(11)

n_rounds = 3000
game1 = rng.choice(game1_values, p=game1_pmf, size=n_rounds)
game2 = rng.choice(game2_values, p=game2_pmf, size=n_rounds)

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
axes[0].plot(game1, linewidth=0.6, color="tab:blue")
axes[0].set_title(f"game 1 outcomes (kurtosis 1): steady +-$1")
axes[1].plot(game2, linewidth=0.6, color="tab:orange")
axes[1].set_title(f"game 2 outcomes (kurtosis 99): near-silence, rare $10 shocks")
for ax in axes:
    ax.set_ylabel("outcome ($)")
    ax.set_ylim(-11, 11)
    ax.grid(True, alpha=0.3)
axes[1].set_xlabel("round")
plt.suptitle("Same variance, opposite texture: where the spread comes from")
plt.tight_layout()
plt.show()

print(f"sample variances over {n_rounds} rounds: "
      f"game 1 = {game1.var():.3f}, game 2 = {game2.var():.3f}")
print("Note game 2's estimate is off despite the true variance being exactly 1:")
print("its variance lives in ~15 rare shocks, so estimates converge very slowly --")
print("heavy tails make even measuring the variance hard. (Try more rounds!)")

### The kurtosis ladder of standard shapes

In [ ]:
rng = np.random.default_rng(42)

def sample_kurtosis(x):
    z = (x - x.mean()) / x.std()
    return np.mean(z**4)

n = 1_000_000
shapes = {
    "coin +-1 (no tails)": rng.choice([-1.0, 1.0], size=n),
    "uniform (flat)": rng.uniform(-1, 1, size=n),
    "normal (reference)": rng.normal(0, 1, size=n),
    "Laplace (thick tails)": rng.laplace(0, 1, size=n),
}

print(f"{'shape':<22} {'kurtosis':>9}   (exact: 1, 1.8, 3, 6)")
for name, x in shapes.items():
    print(f"{name:<22} {sample_kurtosis(x):>9.3f}")

### Kurtosis sees what variance cannot

Mix a narrow normal with occasional wild values, tuned to keep variance = 1: the variance stays blind, the kurtosis explodes.

In [ ]:
rng = np.random.default_rng(42)

n = 1_000_000
outlier_prob = 0.01
# 99% from a narrow normal, 1% from a wide one; scale chosen so total variance = 1
narrow_var = 0.5
wide_var = (1 - (1 - outlier_prob) * narrow_var) / outlier_prob   # = 50.5
is_outlier = rng.random(n) < outlier_prob
mixture = np.where(is_outlier,
                   rng.normal(0, np.sqrt(wide_var), size=n),
                   rng.normal(0, np.sqrt(narrow_var), size=n))

pure = rng.normal(0, 1, size=n)

for name, x in [("pure normal", pure), ("1% outlier mixture", mixture)]:
    print(f"{name:<20} var = {x.var():.3f}   kurtosis = {sample_kurtosis(x):>7.2f}")
print("\nIdentical variance; the mixture's kurtosis screams about its hidden tail.")

## 5. Visualizations

### The two games as distributions

Game 2's bars near zero tower over everything, while its \$10 bars are almost invisible — log scale on the right makes the tiny-but-fatal tails visible.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].bar([-1, 1], [0.5, 0.5], width=0.35, color="tab:blue", edgecolor="black")
axes[0].set_title("game 1: +-$1 with prob 1/2\nkurtosis = 1")
axes[0].set_ylabel("probability")

axes[1].bar([-10, -0.1, 0.1, 10], np.array([1, 100, 100, 1]) / 202,
            width=0.35, color="tab:orange", edgecolor="black")
axes[1].set_yscale("log")
axes[1].set_title("game 2: mostly +-\\$0.10, rarely +-\\$10\nkurtosis = 99  (log scale!)")

for ax in axes:
    ax.set_xlabel("outcome ($)")
    ax.set_xlim(-11.5, 11.5)
    ax.grid(True, alpha=0.3)
plt.suptitle("Same mean (0), variance (1), skewness (0) -- different fourth moment")
plt.tight_layout()
plt.show()

### Thin vs. thick tails at equal variance

A normal and a Laplace distribution scaled to the same variance: the Laplace is *sharper in the middle* and *heavier in the tails* — the trade the variance cannot see. The log-scale panel shows the tails diverging.

In [ ]:
x = np.linspace(-6, 6, 1000)
normal_pdf = np.exp(-0.5 * x**2) / np.sqrt(2 * np.pi)
b = 1 / np.sqrt(2)                      # Laplace scale for variance 1
laplace_pdf = np.exp(-np.abs(x) / b) / (2 * b)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, log in [(axes[0], False), (axes[1], True)]:
    ax.plot(x, normal_pdf, color="tab:blue", linewidth=2,
            label="normal (kurtosis 3)")
    ax.plot(x, laplace_pdf, color="tab:red", linewidth=2,
            label="Laplace (kurtosis 6)")
    if log:
        ax.set_yscale("log")
        ax.set_ylim(1e-8, 1)
        ax.set_title("log scale: the thick tail dominates far out")
    else:
        ax.set_title("linear scale: same variance = 1")
    ax.set_xlabel("value")
    ax.set_ylabel("density")
    ax.grid(True, alpha=0.3)
    ax.legend()
plt.suptitle("Kurtosis compares tails at equal variance")
plt.tight_layout()
plt.show()

### The four descriptors side by side

A closing dashboard: four sample distributions and their (mean, std, skewness, kurtosis) — each descriptor picks up a different feature.

In [ ]:
rng = np.random.default_rng(42)

n = 300_000
datasets = {
    "normal(0, 1)": rng.normal(0, 1, size=n),
    "shifted normal(3, 1)": rng.normal(3, 1, size=n),
    "exponential (skewed)": rng.exponential(1, size=n) - 1,
    "Laplace (thick tails)": rng.laplace(0, 1 / np.sqrt(2), size=n),
}

fig, axes = plt.subplots(1, 4, figsize=(15, 3.6), sharey=True)
for ax, (name, data) in zip(axes, datasets.items()):
    z = (data - data.mean()) / data.std()
    ax.hist(data, bins=120, density=True, color="tab:blue", alpha=0.7)
    ax.set_title(f"{name}\nmean {data.mean():.1f}, std {data.std():.1f},"
                 f"\nskew {np.mean(z**3):.1f}, kurt {np.mean(z**4):.1f}",
                 fontsize=10)
    ax.set_xlim(-6, 6)
    ax.grid(True, alpha=0.3)
axes[0].set_ylabel("density")
plt.suptitle("Mean: center | std: spread | skewness: asymmetry | kurtosis: tails")
plt.tight_layout()
plt.show()

## 6. Exercises

### Basic

**Exercise 1.** Compute the kurtosis of the fair ±\$1 coin directly from the definition. Why is a symmetric two-point distribution the *minimum*-kurtosis shape (what tails does it have)?

**Exercise 2.** For the fair die, compute the kurtosis. Between the coin (1) and the normal (3) — where does it land, and does that match its shape?

### Intermediate

**Exercise 3.** Rebuild game 2 with the ±\$10 outcomes replaced by ±\$5 (keep probabilities $\tfrac{1}{202}$; adjust nothing else). Recompute variance and kurtosis. Both drop — but which one collapses faster, and why (think powers)?

**Exercise 4.** Kurtosis, like skewness, is scale-free: verify numerically that $\mathrm{kurt}(aX + b) = \mathrm{kurt}(X)$ for the exponential distribution with several $a, b$ (including negative $a$ — does the sign flip like skewness did?).

### Advanced

**Exercise 5.** Student's t-distribution with $\nu$ degrees of freedom has thicker tails for smaller $\nu$ (kurtosis $= 3 + \frac{6}{\nu - 4}$ for $\nu > 4$). Estimate the kurtosis by simulation for $\nu = 5, 10, 30$ and compare with the formula. What happens as $\nu \to \infty$?

**Exercise 6.** There is a hard floor: for any distribution, kurtosis $\ge 1 + \text{skewness}^2$ (so kurtosis $\ge 1$ always). Verify the inequality numerically on every distribution used in this notebook (both games, die, uniform, normal, Laplace, exponential). Which one sits exactly on the floor?

## 7. Solutions

In [ ]:
def pmf_kurtosis(values, pmf):
    mu = np.sum(pmf * values)
    sigma = np.sqrt(np.sum(pmf * (values - mu)**2))
    return np.sum(pmf * ((values - mu) / sigma)**4)

def pmf_skewness(values, pmf):
    mu = np.sum(pmf * values)
    sigma = np.sqrt(np.sum(pmf * (values - mu)**2))
    return np.sum(pmf * ((values - mu) / sigma)**3)

# Exercise 1: the +-1 coin
values = np.array([-1.0, 1.0])
pmf = np.array([0.5, 0.5])
print(f"Exercise 1: kurtosis = {pmf_kurtosis(values, pmf)}")
print("Standardized, the coin's outcomes are exactly +-1, so the 4th powers are")
print("exactly 1: no value ever strays from one standard deviation. Zero tail")
print("mass -- no distribution can concentrate deviations tighter than that.")

In [ ]:
# Exercise 2: the fair die
faces = np.arange(1, 7).astype(float)
pmf6 = np.full(6, 1 / 6)
print(f"Exercise 2: kurtosis = {pmf_kurtosis(faces, pmf6):.4f}")
print("About 1.73 -- between the coin (1) and the normal (3), close to the")
print("continuous uniform (1.8): a flat, tail-less shape.")

In [ ]:
# Exercise 3: game 2 with +-$5 instead of +-$10
for extreme in [10.0, 5.0]:
    values = np.array([-extreme, -0.1, 0.1, extreme])
    pmf = np.array([1, 100, 100, 1]) / 202
    mu = np.sum(pmf * values)
    var = np.sum(pmf * (values - mu)**2)
    print(f"extreme = +-${extreme:.0f}: variance = {var:.4f}, "
          f"kurtosis = {pmf_kurtosis(values, pmf):.2f}")
print("\nExercise 3: halving the extreme cut the variance ~4x (2nd power: 1 ->")
print("0.257) and the RAW 4th moment ~16x. But the KURTOSIS barely moved (99 ->")
print("93): being standardized, it re-measures the tails against the new, smaller")
print("sigma. Kurtosis tracks the tail-vs-body PROPORTION, not absolute size --")
print("the game still spends almost all its variance on the rare extreme.")

In [ ]:
# Exercise 4: kurtosis is scale-free
rng = np.random.default_rng(0)
x = rng.exponential(1, size=1_000_000)

def sample_kurtosis(data):
    z = (data - data.mean()) / data.std()
    return np.mean(z**4)

print(f"Exercise 4: kurt(X)         = {sample_kurtosis(x):.3f}")
print(f"            kurt(100X + 7)  = {sample_kurtosis(100 * x + 7):.3f}")
print(f"            kurt(-3X)       = {sample_kurtosis(-3 * x):.3f}")
print("All identical -- and unlike skewness, reflection does NOT flip the sign:")
print("the 4th power erases signs, so kurtosis is always positive.")

In [ ]:
# Exercise 5: Student's t tails
rng = np.random.default_rng(0)

print("Exercise 5:")
print(f"{'nu':>4} {'simulated':>10} {'formula':>9}")
for nu in [5, 10, 30]:
    t = rng.standard_t(nu, size=2_000_000)
    formula = 3 + 6 / (nu - 4)
    print(f"{nu:>4} {sample_kurtosis(t):>10.2f} {formula:>9.2f}")
print("As nu grows the t-distribution loses its heavy tails and the kurtosis")
print("approaches the normal's 3. (Small-nu estimates are noisy -- the very tail")
print("events being measured are rare and huge.)")

In [ ]:
# Exercise 6: the kurtosis floor
rng = np.random.default_rng(0)

pmf_cases = {
    "game 1 (coin)": (np.array([-1.0, 1.0]), np.array([0.5, 0.5])),
    "game 2": (np.array([-10.0, -0.1, 0.1, 10.0]), np.array([1, 100, 100, 1]) / 202),
    "die": (np.arange(1, 7).astype(float), np.full(6, 1 / 6)),
}
sample_cases = {
    "uniform": rng.uniform(0, 1, size=1_000_000),
    "normal": rng.normal(size=1_000_000),
    "Laplace": rng.laplace(size=1_000_000),
    "exponential": rng.exponential(size=1_000_000),
}

print("Exercise 6:")
print(f"{'distribution':<15} {'kurtosis':>9} {'1 + skew^2':>11}")
for name, (v, p) in pmf_cases.items():
    print(f"{name:<15} {pmf_kurtosis(v, p):>9.2f} {1 + pmf_skewness(v, p)**2:>11.2f}")
for name, data in sample_cases.items():
    z = (data - data.mean()) / data.std()
    print(f"{name:<15} {np.mean(z**4):>9.2f} {1 + np.mean(z**3)**2:>11.2f}")
print("\nEvery distribution satisfies kurtosis >= 1 + skew^2. The +-1 coin sits")
print("EXACTLY on the floor (1 >= 1): symmetric two-point distributions are the")
print("extreme case of the inequality.")

## 8. Conclusion

- Game 1 (±\$1) and game 2 (±\$0.10 usually, ±\$10 rarely) agree on the **mean (0)**, the **variance (1)** — carefully arranged via the $\tfrac{100}{202}, \tfrac{1}{202}$ probabilities — and the **skewness (0)**, since both are symmetric.
- The **fourth moment** finally separates them: $E[X_1^4] = 1$ vs. $E[X_2^4] \approx 99.01$, powered by $(\pm10)^4 = 10{,}000$ — the fourth power makes rare far-away values impossible to hide.
- **Kurtosis** is the standardized version, $E\big[\big(\frac{X-\mu}{\sigma}\big)^4\big]$: thin tails → small kurtosis, thick tails → large kurtosis, even at identical variance. (Benchmarks: coin 1, uniform 1.8, normal 3, Laplace 6, game 2 ≈ 99.)
- Kurtosis is scale-free and — unlike skewness — always positive: fourth powers erase signs.
- **The complete toolkit:** expected value (center), variance/standard deviation (spread), skewness (asymmetry), kurtosis (tails). One descriptor per moment, and together they say a lot about any distribution.

This concludes the descriptors of a single distribution. Next, the course turns to **several variables at once**: joint, marginal, and conditional distributions, and covariance.